# Plot Back-slip inversion of the GNSS displacement measurements of interseismic locking (coupling) at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution



In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/coseis/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
flag_savefig = True
# flag_savefig = False

# Define the inversion results path
resultpath = "rst_coseis/"

# define mesh name
# meshname = "nicoya"
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshname = "nicoyaCK4"   # same as CK2, but connecting the trench now

# Meshes with even top boundary at 0 depth
# meshname = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshname = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
# meshname = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
meshname = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshname

print(meshname)

m2mm = 1e3  # meter to mm
km2m = 1e3   # km to m

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
print(trench.head())

# read in the lon and lat of stations
obs_file = "data/Protti_etal_2014_tableS1.csv"
# note that the height is in m, duration and dates are in yr, and the displacements and errors are already in m
# From BAGA to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                   names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                          'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

# displacement noise standard deviations, in m 
error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())


In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region2=[-86.75, -84.4, 8.8, 11.15]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]


In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1

# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# # the optimal weight combination comes from the L-curve test
# # rho_s = 1e7
# rho_s = 1e8
# # rho_s = 1e9

# # gamma_val_H1 = 1e2
# # gamma_val_H1 = 4e2
# gamma_val_H1 = 5e2
# # gamma_val_H1 = 1e3

# delta_val_L2 = gamma_val_H1 / rho_s 

# gamma_val_H1 = 1e2  
# delta_val_L2 = 5e-6

# # gamma_val_H1 = 2e2  
# # delta_val_L2 = 1e-5

# # gamma_val_H1 = 2e2  
# # delta_val_L2 = 5e-6

# # gamma_val_H1 = 5e2  
# # delta_val_L2 = 5e-6

# # preferred damping for unconstrained inversion on the extended fault
# if meshname == "nicoyaCK3":   # fault zone extended to the whole subduction zone
#     # gamma_val_H1 = 1e2  
#     # # delta_val_L2 = 2.5e-6
#     # delta_val_L2 = 5e-6

#     # gamma_val_H1 = 2.5e2  
#     gamma_val_H1 = 5e2  
#     rho_s = 2e7
#     delta_val_L2 = gamma_val_H1 / rho_s 

# elif meshname == "nicoyaCK4":   # smaller fault
#     gamma_val_H1 = 1e2  
#     delta_val_L2 = 5e-6
    

# newest preferred values for the dense mesh, as of 12/08/2025
rho_s = 2e7   # allows variations of slip of the order of ~4.5 km, close to the maximum resolution
# rho_s = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution
# gamma_val_H1 = 6e2
# rho_s = 1e9   # allows variations of slip of the order of ~30 km 
# gammas_s = [1e2, 1.5e2, 2e2, 2.5e2, 3e2] 
gamma_val_H1 = 2.5e2    #2.5e2 leads to very similar potency to KN16
# gamma_val_H1 = 1.5e2
# gamma_val_H1 = 4e2
delta_val_L2 = gamma_val_H1 / rho_s 

gamma_val_H1_sw = 3e2   # to match the slip smoothness with Hom
delta_val_L2_sw = gamma_val_H1_sw / rho_s 

if rho_s == 2e7:
    inv_str = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
    inv_strsw = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1_sw:.1e}_ds{delta_val_L2_sw:.1e}"
elif rho_s == 2.5e8: 
    inv_str = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
    inv_strsw = inv_str

print(meshname)
print(inv_str)
print(inv_strsw)

In [ ]:
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot

    return x_rot, y_rot

# Load the ground-truth displacement for each forward structure model
def read_obs_disp(df, datadir, resultpath, meshname, inv_str):
    outFileName = 'd_obs_' + meshname + inv_str + '.txt'
    print(outFileName)
    u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    u_obs['lon'], u_obs['lat'] = df['lon'], df['lat']  # add lon and lat for merging later    
    # the saved obs disp are aligned with mesh, reverse rotation, back to geo
    u_obs['ux'], u_obs['uy'] = rot_xy(u_obs['ux'], u_obs['uy'], -rot) 
    # vector magnitude
    u_obs['mag'] = np.sqrt(u_obs['ux']**2 + u_obs['uy']**2 + u_obs['uz']**2)
    u_obs['mag_h'] = np.sqrt(u_obs['ux']**2 + u_obs['uy']**2)
    u_obs['mag_v'] = u_obs['uz'].abs()
    # print(u_obs.head())

    return u_obs

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(u_obs, scaleunit, error_e=None, error_n=None, error_v=None):
    # convert from m to mm, horizontal components
    df_obs_h_data = {
        "x": u_obs['lon'],
        "y": u_obs['lat'],
        "east_velocity": u_obs['ux']*scaleunit,
        "north_velocity": u_obs['uy']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"] = error_e*scaleunit
        df_obs_h_data["north_sigma"] = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0
    
    df_obs_h = pd.DataFrame(data=df_obs_h_data)

    # convert from m to mm, vertical components
    df_obs_v_data = {
        "x": u_obs['lon'],
        "y": u_obs['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_obs['uz']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_v is not None:
        df_obs_v_data["east_sigma"] = error_v*scaleunit
        df_obs_v_data["north_sigma"] = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0
    
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    
    return df_obs_h, df_obs_v

In [ ]:
# read in the observation disp.
u_obs = read_obs_disp(df, datadir, resultpath, meshname, inv_str)

# buil observation disp. vectors
df_obs_h, df_obs_v = build_disp_vector(u_obs, m2mm, error_e, error_n, error_v)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
# mu_l = -0.9730  # ~25 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = 0.9730 # ~55 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # string for the contrast between slab and wedge
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the 3d model, value multiplied by 4, relative a reference
    # contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
    contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026
    if not use_uneven_mesh:
        het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"
        het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}"

    else:
        # het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
        # het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull_test1"    # for testing
        het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"    # amplify rel. to 1D 
        het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}_hull"
        het3d_ori_str1 = f"_DeShon3D_ref1D_{round(1.0)}_hull"
        # het3d4_str = f"_DeShon3D_ref1D_{round(4.0)}_hull"    # amplify rel. to 1D 

        het1d_str = f"_DeShon1Dref"  # 1D depth-layered model

        CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
        if CG_mu_deg == 2:
            het3d_str = het3d_str + f"_CG{CG_mu_deg}"
            het3d_ori_str1 = het3d_ori_str1 + f"_CG{CG_mu_deg}"
            het3d_ori_str = het3d_ori_str + f"_CG{CG_mu_deg}"
            # het3d4_str = het3d4_str + f"_CG{CG_mu_deg}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(sw_str)
print(het1d_str)
print(het3d_ori_str)
print(het3d_str)
# print(het3d4_str)

# # define the structure model strings for the INVERSION 
# struc_str_inv_het = anomaly_str
# print("Inverse problem based on: ", struc_str_inv_het)

# struc_str_inv_hom = homo_str
# print("Inverse problem based on: ", struc_str_inv_hom)



In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f/km2m

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
# loc_f = loc_f.iloc[::-1].reset_index(drop=True)
print(loc_f[['lon', 'lat']].head())
# fault.head()


In [ ]:
# Load the predicted surface displacement, all in meters
def read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + inv_str + struc_str_inv + '.txt'
    # print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    u_pred['lon'], u_pred['lat'] = u_obs['lon'], u_obs['lat']  # add lon and lat for merging later    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot) 
    u_pred['mag'] = np.sqrt(u_pred['ux']**2 + u_pred['uy']**2 + u_pred['uz']**2)
    u_pred['mag_h'] = np.sqrt(u_pred['ux']**2 + u_pred['uy']**2)
    u_pred['mag_v'] = u_pred['uz'].abs()

    u_res = u_pred.copy()
    u_res['ux'] = u_obs['ux'] - u_pred['ux']
    u_res['uy'] = u_obs['uy'] - u_pred['uy']
    u_res['uz'] = u_obs['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    u_res['mag_h'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2)
    u_res['mag_v'] = u_res['uz'].abs()
    # u_res.head()

    return u_pred, u_res

# Load the inferred slip on the fault, all in meters
def read_inferred_slip(datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])
    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)

    # print(m_s['s_strike'].min(), m_s['s_strike'].max())
    # print(m_s['s_dip'].min(), m_s['s_dip'].max())
    print(m_s['mag'].min(), m_s['mag'].max())

    return m_s

# Load the inferred moment and potency of the slip
def read_pred_moment(datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'moment_' + meshname + inv_str + struc_str_inv + '.txt'
    # print(outFileName)
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['moment', 'mw', 'potency'])
    print(mom.head())
    # print("Inferred moment (Nm): ", mom['moment'].values[0])
    # print("Inferred moment magnitude Mw: ", mom['Mw'].values[0])
    # print("Inferred potency (m^3): ", mom['potency'].values[0])

    return mom

In [ ]:
##### Load the results of the homogeneous inversion
# Load the predicted surface displacement, all in meters
u_pred_hom, u_res_hom = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, homo_str)
# Load the inferred slip on the fault, all in meters
m_s_hom = read_inferred_slip(datadir, resultpath, meshname, inv_str, homo_str)
mom_hom = read_pred_moment(datadir, resultpath, meshname, inv_str, homo_str)


In [ ]:
##### Load the results of the slab/wedge model inversion
# inv_strsw = '_coseis_w11_gs8.0e+01_ds3.2e-07'
# inv_strsw = '_coseis_w11_gs2.0e+02_ds8.0e-07'
# inv_strsw = '_coseis_w11_gs4.0e+02_ds1.6e-06'
# inv_strsw = '_coseis_w11_gs6.0e+02_ds2.4e-06'
# inv_strsw = '_coseis_w11_gs8.0e+02_ds3.2e-06'
# inv_strsw = inv_str
u_pred_sw, u_res_sw = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_strsw, sw_str)
m_s_sw = read_inferred_slip(datadir, resultpath, meshname, inv_strsw, sw_str)
mom_sw = read_pred_moment(datadir, resultpath, meshname, inv_strsw, sw_str)

# if rho_s == 2e7:
#     if gamma_val_H1 == 6e2:
#         inv_str3d = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
#         inv_str3d = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
#     else:
#         inv_str3d = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
#         inv_str3d = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"

if rho_s == 2e7:
    inv_str3d = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
elif rho_s == 2.5e8:
    inv_str3d = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"

##### Load the results of the depth-layered 1d model inversion
u_pred_1d, u_res_1d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str3d, het1d_str)
m_s_1d = read_inferred_slip(datadir, resultpath, meshname, inv_str3d, het1d_str)
mom_1d = read_pred_moment(datadir, resultpath, meshname, inv_str3d, het1d_str)

##### Load the results of the original 3d model inversion
u_pred_3d_ori, u_res_3d_ori = read_pred_disp(u_obs, datadir, resultpath, meshname, 
                                             inv_str3d, het3d_ori_str)
m_s_3d_ori = read_inferred_slip(datadir, resultpath, meshname, inv_str3d, het3d_ori_str)
mom_3d_ori = read_pred_moment(datadir, resultpath, meshname, inv_str3d, het3d_ori_str)

##### Load the results of the amplified 3d model inversion, x2.5
u_pred_3d, u_res_3d = read_pred_disp(u_obs, datadir, resultpath, meshname, 
                                            inv_strsw, het3d_str)
m_s_3d = read_inferred_slip(datadir, resultpath, meshname, inv_strsw, het3d_str)
mom_3d = read_pred_moment(datadir, resultpath, meshname, inv_strsw, het3d_str)

# ##### Load the results of the amplified 3d model inversion, x4
# u_pred_3d4, u_res_3d4 = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str3d, het3d4_str)
# m_s_3d4 = read_inferred_slip(datadir, resultpath, meshname, inv_str3d, het3d4_str)
# mom_3d4 = read_pred_moment(datadir, resultpath, meshname, inv_str3d, het3d4_str)

# difference between the heterogeneous and homogeneous model
# m_s_hom['mag'].max()
# m_s_sw['mag'].max()
# m_s_3d_ori['mag'].max()
# m_s_3d['mag'].max()

print((m_s_sw['mag']-m_s_hom['mag']).min(), (m_s_sw['mag']-m_s_hom['mag']).max())
print((m_s_1d['mag']-m_s_hom['mag']).min(), (m_s_1d['mag']-m_s_hom['mag']).max())
print((m_s_3d_ori['mag']-m_s_hom['mag']).min(), (m_s_3d_ori['mag']-m_s_hom['mag']).max())
print((m_s_3d['mag']-m_s_hom['mag']).min(), (m_s_3d['mag']-m_s_hom['mag']).max())
# print((m_s_3d4['mag']-m_s_hom['mag']).min(), (m_s_3d4['mag']-m_s_hom['mag']).max())


In [ ]:
# Load the inferred interseismic locking ratio on the fault, all in meters
def read_inferred_locking(datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])
    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)

    cols_to_convert = ['s_strike', 's_dip', 'mag']
    m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
    # print(m_s['s_strike'].min(), m_s['s_strike'].max())
    # print(m_s['s_dip'].min(), m_s['s_dip'].max())
    # print(m_s['mag'].max())

    return m_s

# Define the inversion results path
resultpath_lock = "rst_locking/"

# # preferred damping for the unconstrained inversion 
# if meshname == "nicoyaCK3":   # fault zone extended to the whole subduction zone
#     gamma_val_H1_lock = 2.5e3
#     delta_val_L2_lock = 1e-5
# elif meshname == "nicoyaCK4":   # smaller fault
#     gamma_val_H1_lock = 2.5e3  
#     delta_val_L2_lock = 2.5e-5

# newest preferred values for the dense mesh, as of 12/08/2025
rho_s_lock = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution
gamma_val_H1_lock = 3e3
delta_val_L2_lock = gamma_val_H1_lock / rho_s_lock 

inv_str_lock = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1_lock:.0e}_ds{delta_val_L2_lock:.0e}"
print(inv_str_lock)

gamma_val_H1_lock_ampl = 3e3
delta_val_L2_lock_ampl = gamma_val_H1_lock_ampl / rho_s_lock 
inv_str_lock_ampl = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1_lock_ampl:.0e}_ds{delta_val_L2_lock_ampl:.0e}"

# Load the inferred slip on the fault, all in meters
m_s_hom_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock, homo_str)
##### Load the results of the slab/wedge model inversion
m_s_sw_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock, sw_str)
##### Load the results of the depth-layered 1d model inversion
m_s_1d_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock, het1d_str)
##### Load the results of the original 3d model inversion
m_s_3d_ori_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock, het3d_ori_str)
##### Load the results of the amplified 3d model inversion, x2.5
m_s_3d_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock_ampl, het3d_str)
# ##### Load the results of the amplified 3d model inversion, x4
# m_s_3d4_lock = read_inferred_locking(datadir, resultpath_lock, meshname, inv_str_lock_ampl, het3d4_str)

# define interseismic coupling between the two plates as the ratio of back normal slip rate
# (m_est vector) to local trench-normal convergence rate (m_est/V_norm.)
V_norm = 78.5   # trench-normal of Cocos-Caribbean motion, mm/yr
V_const_para = 11     # only remove the a constant value from trench parallel component, mm/yr 
V_para = 27-V_const_para    # trench-parallel of Cocos-Caribbean motion, mm/yr   

def interseismic_coupling(m_s, V_norm):
    m_s['locking_dip'] = m_s['s_dip']/V_norm
    m_s['locking'] = m_s['mag']/ np.sqrt(V_norm**2+V_para**2)

    return m_s

m_s_hom_lock = interseismic_coupling(m_s_hom_lock, V_norm)
m_s_sw_lock = interseismic_coupling(m_s_sw_lock, V_norm)
m_s_1d_lock = interseismic_coupling(m_s_1d_lock, V_norm)
m_s_3d_ori_lock = interseismic_coupling(m_s_3d_ori_lock, V_norm)
m_s_3d_lock = interseismic_coupling(m_s_3d_lock, V_norm)
# m_s_3d4_lock = interseismic_coupling(m_s_3d4_lock, V_norm)

print(m_s_hom_lock['locking'].min(), m_s_hom_lock['locking'].max())
print(m_s_sw_lock['locking'].min(), m_s_sw_lock['locking'].max())
print(m_s_1d_lock['locking'].min(), m_s_1d_lock['locking'].max())
print(m_s_3d_ori_lock['locking'].min(), m_s_3d_ori_lock['locking'].max())
print(m_s_3d_lock['locking'].min(), m_s_3d_lock['locking'].max())
# print(m_s_3d4_lock['locking'].min(), m_s_3d4_lock['locking'].max())

print((m_s_sw_lock['locking']-m_s_hom_lock['locking']).min(), (m_s_sw_lock['locking']-m_s_hom_lock['locking']).max())
print((m_s_1d_lock['locking']-m_s_hom_lock['locking']).min(), (m_s_1d_lock['locking']-m_s_hom_lock['locking']).max())
print((m_s_3d_ori_lock['locking']-m_s_hom_lock['locking']).min(), (m_s_3d_ori_lock['locking']-m_s_hom_lock['locking']).max())
print((m_s_3d_lock['locking']-m_s_hom_lock['locking']).min(), (m_s_3d_lock['locking']-m_s_hom_lock['locking']).max())
# print((m_s_3d4_lock['locking']-m_s_hom_lock['locking']).min(), (m_s_3d4_lock['locking']-m_s_hom_lock['locking']).max())

mom_rate_hom = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock, homo_str)
mom_rate_sw = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock, sw_str)
mom_rate_1d = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock, het1d_str)
mom_rate_3d_ori = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock, het3d_ori_str)
mom_rate_3d = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock_ampl, het3d_str)
# mom_rate_3d4 = read_pred_moment(datadir, resultpath_lock, meshname, inv_str_lock_ampl, het3d4_str)


In [ ]:
# Potency comparison: coseismic vs interseismic, relative to Hom (H)
# ─────────────────────────────────────────────────────────────────
# Percentages are derived from the displayed (rounded) values so that
# a reader can reproduce them directly from the table.
labels  = ['Hom (H)', 'SW (S)',  '1D',      'Orig. 3D',    'Ampl. 3D']
mom_c   = [mom_hom,   mom_sw,   mom_1d,    mom_3d_ori,    mom_3d     ]
mom_i   = [mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori, mom_rate_3d]

# Round to .2e precision (2 decimal digits in scientific notation)
pot_c = [float(f"{m['potency'].values[0]:.2e}") for m in mom_c]
pot_i = [float(f"{m['potency'].values[0]:.2e}") for m in mom_i]

hdr = (f"{'Model':12s}  {'Coseis pot (m³)':>16s}  {'vs Hom':>8s}"
       f"  {'Interseis rate (m³/yr)':>22s}  {'vs Hom':>8s}")
sep = '-' * len(hdr)
print(hdr)
print(sep)
for i, lbl in enumerate(labels):
    c_rel = (pot_c[i] / pot_c[0] - 1) * 100
    l_rel = (pot_i[i] / pot_i[0] - 1) * 100
    c_rel_str = f"{c_rel:+.1f}%" if i > 0 else "  (ref)"
    l_rel_str = f"{l_rel:+.1f}%" if i > 0 else "  (ref)"
    print(f"{lbl:12s}  {pot_c[i]:16.2e}  {c_rel_str:>8s}"
          f"  {pot_i[i]:22.2e}  {l_rel_str:>8s}")


In [ ]:
def plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.1c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "14c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
            maxslip = 2.8
            print(maxslip)
            # slipstep = maxslip/20
            slipstep = 0.2
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_hom[col2plt].max():.2f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_hom['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["a0.4f0.2", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the slab/wedge model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tS/W inv."])
            # maxslip = m_s_sw[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_sw[col2plt].max():.2f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_sw['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["a0.4f0.2", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the original 3d model
        with fig.set_panel(panel=[1, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D inv."])
            # maxslip = m_s_3d[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_ori_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_ori[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_3d_ori[col2plt].max():.2f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_3d_ori['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["a0.4f0.2", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the amplified 3d model
        with fig.set_panel(panel=[1, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tAmplified 3D inv."])
            # maxslip = m_s_3d[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_3d[col2plt].max():.2f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_3d['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["a0.4f0.2", "x+lMagnitude", "y+l(m)"]) #


    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_sum_invslip.pdf")

# plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, 'locking', region, flag_savefig=True)
# plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, 'locking2', region, flag_savefig=False)
# plot_all_inferred_slip_(m_s_hom, m_s_hom, m_s_hom, 'mag', region, flag_savefig=False)
# plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d_ori, 'mag', region1, flag_savefig=False)


In [ ]:
def plot_all_inferred_slip2(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region, flag_savefig=False, figname=None):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p",
                    COLOR_BACKGROUND="red", COLOR_FOREGROUND="blue",
                    )
        
        mu_model_strings = ["H", "S", "orig. 3D", "ampl. 3D"]
        
        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[0]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
            # maxslip = 2.8
            maxslip = 3.6
            print(maxslip)
            slipstep = maxslip/10
            # slipstep = 0.2
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_hom[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_hom[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_hom[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_hom['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"M@-0@-:", x=region[0]+0.05, y=region[-2]+0.95, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{mom_hom['moment'].iloc[0]:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.8, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=[f"a{slipstep*5}f{slipstep}", "x+lSlip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #
            
            # Legend box bottom-left corner (adjust to your map's coordinates)
            x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
            # Box size
            width = 0.65    # degrees or projected units depending on your region
            height = 0.35
            # 1 — draw the gray box
            fig.plot(
                x=[x0, x0+width, x0+width, x0, x0],
                y=[y0, y0, y0+height, y0+height, y0],
                pen="0.5p,black",
                fill="lightgray",
                # transparency=30,
            )

            # Manual legend - more control and simpler
            # Position in your subplot coordinates
            x_legend = region[0] + 0.1  # adjust based on your region
            y_legend = region[2] + 0.45

            # Draw black line
            fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
                    pen="1p,black")
            fig.text(x=x_legend+0.15, y=y_legend+0.15, text="0.9 locking",
                    font="5p,Helvetica", justify="LM")

            # Draw white line
            fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
                    pen="1p,white")
            fig.text(x=x_legend+0.15, y=y_legend, text="0.5 locking",
                    font="5p,Helvetica", justify="LM")


        # Slip inferred from the SW model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[1]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
            # maxslip = 2.8
            # print(maxslip)
            # slipstep = maxslip/20
            # slipstep = 0.2
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_sw[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_sw[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_sw[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_sw['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"M@-0@-: {mom_sw['moment'].iloc[0]:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=[f"a{slipstep*5}f{slipstep}", "x+lSlip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #

        # Slip inferred from the 3D model
        with fig.set_panel(panel=[0, 2]):

            m_s_3d_plt = m_s_3d_ori
            mom_3d_plt = mom_3d_ori
            m_s_3d_lock_plt = m_s_3d_ori_lock
            mu_model_num = 2

            # m_s_3d_plt = m_s_3d
            # mom_3d_plt = mom_3d
            # m_s_3d_lock_plt = m_s_3d_lock
            # mu_model_num = 3

            # m_s_3d_plt = m_s_3d
            # mom_3d_plt = mom_3d4
            # m_s_3d_lock_plt = m_s_3d4_lock
            # mu_model_num = 3

            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[mu_model_num]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
            # maxslip = 2.8
            # print(maxslip)
            # slipstep = maxslip/20
            # slipstep = 0.2
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_lock_plt['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_3d_plt[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_3d_plt[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_plt[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_plt[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_3d_plt[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_3d_plt['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"M@-0@-: {mom_3d_plt['moment'].iloc[0]:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=[f"a{slipstep*5}f{slipstep}", "x+lSlip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + figname)

figname = meshname + "_coseis.pdf"
plot_all_inferred_slip2(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d_ori, 'mag', region1, flag_savefig=False, figname=figname)
# plot_all_inferred_slip2(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d4, 'mag', region1, flag_savefig=True, figname=figname)


In [ ]:
def plot_slip_diff(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        mu_model_strings = ["S - H", "orig. 3D - H", "ampl. 3D - H"]

        # # Slip inferred from the homogeneous model
        # with fig.set_panel(panel=[0, 0]):
        #     fig.basemap(region=region, projection="M?", frame=["a1f0.2"])
        #     # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
        #     #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
        #     # maxslip = 1
        #     # maxslip = mtrue_s[col2plt].max()
        #     maxslip = m_s_hom[col2plt].max()
        #     # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
        #     maxslip = 2.8
        #     print(maxslip)
        #     # slipstep = maxslip/20
        #     slipstep = 0.2
        #     pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
        #     # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
        #     grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
        #     # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

        #     fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
        #     fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
        #     fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

        #     fig.text(text=f"Max. slip: {m_s_hom[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
        #     fig.text(text=f"Potency: {mom_hom['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
        #     scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
        #     mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        #     fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        #     grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
        #     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        #     # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
        #     with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #         fig.colorbar(frame=["af", "x+lSlip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #
            
        #     # Legend box bottom-left corner (adjust to your map's coordinates)
        #     x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
        #     # Box size
        #     width = 0.65    # degrees or projected units depending on your region
        #     height = 0.35
        #     # 1 — draw the gray box
        #     fig.plot(
        #         x=[x0, x0+width, x0+width, x0, x0],
        #         y=[y0, y0, y0+height, y0+height, y0],
        #         pen="0.5p,black",
        #         fill="lightgray",
        #         # transparency=30,
        #     )

        #     # Manual legend - more control and simpler
        #     # Position in your subplot coordinates
        #     x_legend = region[0] + 0.1  # adjust based on your region
        #     y_legend = region[2] + 0.45

        #     # Draw black line
        #     fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
        #             pen="1p,black")
        #     fig.text(x=x_legend+0.15, y=y_legend+0.15, text="0.9 locking",
        #             font="5p,Helvetica", justify="LM")

        #     # Draw white line
        #     fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
        #             pen="1p,white")
        #     fig.text(x=x_legend+0.15, y=y_legend, text="0.5 locking",
        #             font="5p,Helvetica", justify="LM")
            
        # difference between the SW heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            m_s = m_s_sw
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[0]}"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.35
            # diffstep = 0.05
            # maxdiff = 0.3
            maxdiff = 0.5
            diffstep = maxdiff/10
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"@~D@~ Slip: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}] m",
                     x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ Potency: {(mom_sw['potency'].iloc[0]-mom_hom['potency'].iloc[0]):.2e} m@+3@+", 
                     x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ M@-0@-: {(mom_sw['moment'].iloc[0]-mom_hom['moment'].iloc[0]):.2e} m@+3@+", 
                     x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")
             
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #

        # difference between the original 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            m_s = m_s_3d_ori
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[1]}"])
            # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_ori_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"@~D@~ Slip: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}] m",
                     x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ Potency: {(mom_3d_ori['potency'].iloc[0]-mom_hom['potency'].iloc[0]):.2e} m@+3@+", 
                     x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ M@-0@-: {(mom_3d_ori['moment'].iloc[0]-mom_hom['moment'].iloc[0]):.2e} m@+3@+", 
                     x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                # fig.colorbar(frame=["af", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #

        # # difference between the amplified 3D heterogeneous and homogeneous model
        # with fig.set_panel(panel=[0, 2]):
        #     # m_s = m_s_3d
        #     # m_s_lock = m_s_3d_lock
        #     # mom = mom_3d

        #     m_s = m_s_3d4
        #     m_s_lock = m_s_3d4_lock
        #     mom = mom_3d4
            
        #     fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[2]}"])
        #     # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
        #     # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
        #     pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
        #     # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
        #     # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
        #     # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
        #     # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
        #     # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
        #     # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
        #     fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            
        #     grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
        #     fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

        #     fig.text(text=f"@~D@~ Slip: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}] m",
        #              x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
        #     fig.text(text=f"@~D@~ Potency: {(mom['potency'].iloc[0]-mom_hom['potency'].iloc[0]):.2e} m@+3@+", 
        #              x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
        #     fig.text(text=f"@~D@~ M@-0@-: {(mom['moment'].iloc[0]-mom_hom['moment'].iloc[0]):.2e} m@+3@+", 
        #              x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")

        #     fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        #     # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
        #     grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
        #     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        #     # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
        #     with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #         # fig.colorbar(frame=["af", "x+lAbs. difference"])
        #         fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}f", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_coseis_diff.pdf")


# plot_slip_diff(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, 'mag', region1, flag_savefig=True)
# plot_slip_diff(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d4, 'mag', region1, flag_savefig=False)
plot_slip_diff(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d_ori, 'mag', region1, flag_savefig=False)


In [ ]:
def plot_inferred_slip_panel(fig, m_s, m_s_lock, mom, panel_idx, title_str, panel_label, 
                             maxslip, slip_step, region, col2plt, slipsybsz, colormap):
    
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
    #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
    # maxslip = 1
    # maxslip = mtrue_s[col2plt].max()
    # maxslip = m_s_hom[col2plt].max()
    # maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max(), m_s_3d_ori[col2plt].max())
    # maxslip = 2.8
    # print(maxslip)
    # slip_step = maxslip/20
    # slip_step = 0.2
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slip_step], background=True, reverse=True)    #m_s_hom[col2plt].max()
    # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
    grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
    # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
    # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation

    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

    fig.text(text=f"Max. slip: {m_s[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency: {mom['potency'].iloc[0]:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

    if panel_idx == 0:
        fig.text(text=f"M@-0@-:", x=region[0]+0.05, y=region[-2]+0.95, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{mom['moment'].iloc[0]:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.8, font="6p,Helvetica,black", justify="ML")
    else:
        fig.text(text=f"M@-0@-: {mom['moment'].iloc[0]:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")

    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
    with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        # fig.colorbar(frame=["af", "x+lCoseismic slip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #
        fig.colorbar(frame=[f"a{slip_step*5}f{slip_step}", "x+lCoseismic slip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c") #

    #plot panel label
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    
    if panel_idx == 0:
        # Legend box bottom-left corner (adjust to your map's coordinates)
        x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
        # Box size
        width = 0.7    # degrees or projected units depending on your region
        height = 0.35
        # 1 — draw the gray box
        fig.plot(
            x=[x0, x0+width, x0+width, x0, x0],
            y=[y0, y0, y0+height, y0+height, y0],
            pen="0.5p,black",
            fill="lightgray",
            # transparency=30,
        )

        # Manual legend - more control and simpler
        # Position in your subplot coordinates
        x_legend = region[0] + 0.1  # adjust based on your region
        y_legend = region[2] + 0.45

        # Draw black line
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
                pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking",
                font="5p,Helvetica", justify="LM")

        # Draw white line
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
                pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking",
                font="5p,Helvetica", justify="LM")
    
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", pen="0.5p,dodgerblue") #, fill="dodgerblue"

def plot_inferred_slip_diff_panel(fig, m_s, m_s_lock, mom, title_str, panel_label, maxdiff, diff_step, 
                                  region, col2plt, slipsybsz, colormap):
    
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
    # maxdiff = 0.16
    # diffstep = 0.02
    # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
    pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
    # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
    # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
    # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
    # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
    # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
    # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
    # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

    m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
    # order = m_s_diff.argsort()
    # fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
    fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

    grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_lock['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

    fig.text(text=f"@~D@~ Slip: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}] m",
                x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"@~D@~ Potency: {(mom['potency'].iloc[0]-mom_hom['potency'].iloc[0]):.2e} m@+3@+", 
                x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"@~D@~ M@-0@-: {(mom['moment'].iloc[0]-mom_hom['moment'].iloc[0]):.2e} m@+3@+", 
                x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")
               
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
    grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
    with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        # fig.colorbar(frame=["af", "x+lAbs. difference"])
        # fig.colorbar(frame=["af", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #
        fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c") #

    #plot panel label
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")



def plot_all_inferred_slip_and_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d, 
                                    m_s_hom_lock, m_s_sw_lock, m_s_1d_lock, m_s_3d_lock,
                                    mom_hom, mom_sw, mom_1d, mom_3d, 
                                    col2plt, region, maxslip, slip_step, maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    colormap_diff = "roma"

    # mu_model_strings = ["H", "S", "3D", "S - H", "3D - H"]
    # mu_model_strings = ["H", "S", "Orig. 3D", "S - H", "Orig. 3D - H"]
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=2, ncols=4, figsize=("20c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_slip_panel(fig, m_s_hom, m_s_hom_lock, mom_hom, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # Slip inferred from the SW model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_sw, m_s_sw_lock, mom_sw, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)
            
        # Slip inferred from the 1D model
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_1d, m_s_1d_lock, mom_1d, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # Slip inferred from the 3D model
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # difference between the SW heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_sw, m_s_sw_lock, mom_sw, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

        # difference between the 1D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 2]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_1d, m_s_1d_lock, mom_1d, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

        # difference between the 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 3]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_coseis_and_diff.pdf"
        figname = meshname + "_ref1D_coseis_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxslip = 2.8
    slip_step = 0.2
    slip_step = maxslip/20

    maxdiff = 0.4
    # diff_step = 0.05
    diff_step = maxdiff/10

else:
    # maxslip = 2.8  # used for gamma=6e2
    maxslip = 3.6  # used for gamma=2.5e2/3e2
    # slip_step = 0.2
    # slip_step = maxslip/20
    slip_step = maxslip/10

    # maxdiff = 0.3  # used for gamma=6e2
    maxdiff = 0.5  # used for gamma=2.5e2/3e2
    # diff_step = 0.05
    diff_step = maxdiff/10

plot_all_inferred_slip_and_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, 
                                m_s_hom_lock, m_s_sw_lock, m_s_1d_lock, m_s_3d_ori_lock,
                                mom_hom, mom_sw, mom_1d, mom_3d_ori, 
                                'mag', region1, maxslip, slip_step, maxdiff, diff_step, 
                                flag_savefig=False)


In [ ]:
# ── v2: shared colorbars in the empty [1,0] panel ──────────────────────────────
# Redefine helpers with optional show_cbar flag (default True = backward compatible)

def plot_inferred_slip_panel(fig, m_s, m_s_lock, mom, panel_idx, title_str, panel_label,
                              maxslip, slip_step, region, col2plt, slipsybsz, colormap, 
                              show_cbar=True, show_m0=False):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slip_step], background=True, reverse=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s_lock["locking"], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt], cmap=True)
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"Max. slip: {m_s[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    _potency = mom["potency"].iloc[0]
    _moment  = mom["moment"].iloc[0]
    fig.text(text=f"Potency: {_potency:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    if show_m0:
        fig.text(text=f"M@-0@-:", x=region[0]+0.05, y=region[-2]+0.95, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{_moment:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.8, font="6p,Helvetica,black", justify="ML")
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=[f"a{slip_step*5}f{slip_step}", "x+lCoseismic slip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    if panel_idx == 0:
        x0, y0 = region[0]+0.05, region[-2]+0.35
        width, height = 0.7, 0.35
        fig.plot(x=[x0, x0+width, x0+width, x0, x0], y=[y0, y0, y0+height, y0+height, y0],
                 pen="0.5p,black", fill="lightgray")
        x_legend = region[0] + 0.1
        y_legend = region[2] + 0.45
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15], pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend], pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=df["lon"], y=df["lat"], style="s0.15c", pen="0.5p,dodgerblue")


def plot_inferred_slip_diff_panel(fig, m_s, m_s_lock, mom, title_str, panel_label, maxdiff, diff_step,
                                   region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
    m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
    order = m_s_diff.argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s_lock["locking"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"@~D@~ Slip: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}] m",
             x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    _dpotency = mom["potency"].iloc[0] - mom_hom["potency"].iloc[0]
    delta_max_slip = float(f"{m_s[col2plt].max():.2f}") - float(f"{m_s_hom[col2plt].max():.2f}")
    fig.text(text=f"@~D@~ Max. slip: {delta_max_slip:+.2f} m",
             x=region[0]+0.05, y=region[-2]+0.40, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"@~D@~ Potency: {_dpotency:.2e} m@+3@+",
             x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lSlip difference", "y+l(m)"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")


def plot_all_inferred_slip_and_diff_v2(m_s_hom, m_s_sw, m_s_1d, m_s_3d,
                                        m_s_hom_lock, m_s_sw_lock, m_s_1d_lock, m_s_3d_lock,
                                        mom_hom, mom_sw, mom_1d, mom_3d,
                                        col2plt, region, maxslip, slip_step, maxdiff, diff_step, flag_savefig=False):
    slipsybsz = "c0.11c"
    colormap = "plasma"
    colormap_diff = "roma"
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=4, figsize=("20c", "11.0c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.0c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_slip_panel(fig, m_s_hom, m_s_hom_lock, mom_hom, panel_idx,
                                     mu_model_strings[panel_idx], panel_labels[panel_idx],
                                     maxslip, slip_step, region, col2plt, slipsybsz, colormap, show_cbar=False, show_m0=True)
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_sw, m_s_sw_lock, mom_sw, panel_idx,
                                     mu_model_strings[panel_idx], panel_labels[panel_idx],
                                     maxslip, slip_step, region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_1d, m_s_1d_lock, mom_1d, panel_idx,
                                     mu_model_strings[panel_idx], panel_labels[panel_idx],
                                     maxslip, slip_step, region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, panel_idx,
                                     mu_model_strings[panel_idx], panel_labels[panel_idx],
                                     maxslip, slip_step, region, col2plt, slipsybsz, colormap, show_cbar=False)

        # ── Empty panel [1,0]: two shared colorbars ──────────────────────────
        with fig.set_panel(panel=[1, 0]):
            with pygmt.config(MAP_FRAME_PEN="0p,white", MAP_TICK_PEN_PRIMARY="0p,white"):
                fig.basemap(region=[0, 1, 0, 1], projection="X?/?", frame="0")
            # Colorbar 1: Slip magnitude
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slip_step], background=True, reverse=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=[f"a{slip_step*5}f{slip_step}", "x+lCoseismic slip magnitude", "y+l(m)"],
                             position="JMC+w3.8c/0.2c+h+o0c/2.4c")
            # Colorbar 2: Slip difference
            pygmt.makecpt(cmap=colormap_diff, series=[-maxdiff, maxdiff, diff_step], background=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lSlip difference", "y+l(m)"],
                             position="JMC+w3.8c/0.2c+h+o0c/1.2c")

        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_sw, m_s_sw_lock, mom_sw,
                                          mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 2]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_1d, m_s_1d_lock, mom_1d,
                                          mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 3]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_3d, m_s_3d_lock, mom_3d,
                                          mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff, show_cbar=False)

    fig.show()
    if flag_savefig:
        figname = meshname + "_ref1D_coseis_and_diff_v2.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

plot_all_inferred_slip_and_diff_v2(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
                                    m_s_hom_lock, m_s_sw_lock, m_s_1d_lock, m_s_3d_ori_lock,
                                    mom_hom, mom_sw, mom_1d, mom_3d_ori,
                                    "mag", region1, maxslip, slip_step, maxdiff, diff_step, 
                                    flag_savefig=flag_savefig)


In [ ]:
def plot_all_inferred_slip_and_diff_extra(m_s_3d, m_s_3d_lock, mom_3d, 
                                          col2plt, region, maxslip, slip_step, 
                                          maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    colormap_diff = "roma"

    mu_model_strings = ["Ampl. 3D", "Ampl. 3D - H"]

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]
    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0

        # Slip inferred from the amplified 3D model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_slip_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # difference between the amplified 3D and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_coseis_and_diff.pdf"
        figname = meshname + "_ref1D_suppl_coseis_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxslip = 2.8
    slip_step = 0.2
    slip_step = maxslip/20

    maxdiff = 0.4
    # diff_step = 0.05
    diff_step = maxdiff/10

else:
    # maxslip = 2.8  # used for gamma=6e2
    maxslip = 3.6  # used for gamma=2.5e2/3e2
    # slip_step = 0.2
    # slip_step = maxslip/20
    slip_step = maxslip/10

    # maxdiff = 0.3  # used for gamma=6e2
    maxdiff = 0.5  # used for gamma=2.5e2/3e2
    # diff_step = 0.05
    diff_step = maxdiff/10

plot_all_inferred_slip_and_diff_extra(m_s_3d, m_s_3d_lock, mom_3d, 
                                'mag', region1, maxslip, slip_step, maxdiff, diff_step, 
                                flag_savefig=flag_savefig)


In [ ]:
def plot_inferred_locking_panel(fig, m_s, mom_rate, panel_idx, title_str, panel_label, 
                                region, col2plt, slipsybsz, rgb_list, show_cbar=True,
                                reverse_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    maxslip = 1
    slipstep = 0.1
    # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=reverse_cbar)
    pygmt.makecpt(cmap=",".join(rgb_list), series=[0, maxslip, slipstep], reverse=reverse_cbar)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
#     fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt], cmap=True)
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"Max. locking: {m_s[col2plt].max():.2f}", x=region[0]+0.05, y=region[-2]+0.1,
             font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency rate: {mom_rate[potency_col].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05, 
             y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    if panel_idx == 0:
        x0, y0 = region[0]+0.05, region[-2]+0.35
        width = 0.7
        height = 0.35
        fig.plot(x=[x0, x0+width, x0+width, x0, x0], y=[y0, y0, y0+height, y0+height, y0],
                 pen="0.5p,black", fill="lightgray")
        x_legend = region[0] + 0.1
        y_legend = region[2] + 0.45
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15], pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend], pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking", font="5p,Helvetica", justify="LM")
        
        
def plot_inferred_slip_panel(fig, m_s, m_s_lock, mom, panel_idx, title_str, panel_label,
                              maxslip, slip_step, region, col2plt, slipsybsz, colormap, 
                              show_cbar=True, reverse_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slip_step], background=True, reverse=reverse_cbar)
    pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=reverse_cbar)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s_lock["locking"], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt], cmap=True)
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"Max. slip: {m_s[col2plt].max():.2f} m", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    _potency = mom["potency"].iloc[0]
    _moment  = mom["moment"].iloc[0]
    fig.text(text=f"Potency: {_potency:.2e} m@+3@+", x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    if panel_idx == 0:
        fig.text(text=f"M@-0@-:", x=region[0]+0.05, y=region[-2]+0.95, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{_moment:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.8, font="6p,Helvetica,black", justify="ML")
    else:
        fig.text(text=f"M@-0@-: {_moment:.2e} Nm", x=region[0]+0.05, y=region[-2]+0.4, font="6p,Helvetica,black", justify="ML")
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=[f"a{slip_step*5}f{slip_step}", "x+lCoseismic slip magnitude", "y+l(m)"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    if panel_idx == 0:
        x0, y0 = region[0]+0.05, region[-2]+0.35
        width, height = 0.7, 0.35
        fig.plot(x=[x0, x0+width, x0+width, x0, x0], y=[y0, y0, y0+height, y0+height, y0],
                 pen="0.5p,black", fill="lightgray")
        x_legend = region[0] + 0.1
        y_legend = region[2] + 0.45
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15], pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend], pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=df["lon"], y=df["lat"], style="s0.15c", pen="0.5p,dodgerblue")

def plot_inferred_locking_and_slip(m_s_3d_lock, mom_rate_3d, m_s_3d, mom_3d,
                                   col2plt_lock, col2plt, region, maxslip, slip_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    colormap_diff = "roma"

    # 10 precise RGB triplets extracted from your colorbar image, extracted from KN16 fig 6b
    rgb_list = [
        "255/18/250",   # 0.0-0.1 (Bright Purple/Magenta)
        "101/15/248",     # 0.1-0.2 (Blue)
        "0/15/246",   # 0.2-0.3 (Azure/Light Blue)
        "0/121/187",   # 0.3-0.4 (Cyan)
        "0/255/253",   # 0.4-0.5 (Spring Green)
        "0/253/48",     # 0.5-0.6 (Pure Green)
        "0/254/48",   # 0.6-0.7 (Chartreuse/Lime)
        "178/254/46",   # 0.7-0.8 (Yellow)
        "255/161/60",   # 0.8-0.9 (Orange)
        "255/20/29"      # 0.9-1.0 (Red)
        ]

    mu_model_strings = ["Interseismic locking ratio", "Coseismic slip magnitude"]

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]
    panel_labels = []

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0

        # Slip inferred from the amplified 3D model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_3d_lock, mom_rate_3d, panel_idx, 
                                        mu_model_strings[panel_idx], '', 
                                        region, col2plt_lock, slipsybsz, rgb_list,
                                        reverse_cbar=False)


        # difference between the amplified 3D and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, panel_idx, 
                                     mu_model_strings[panel_idx], '', 
                                     maxslip, slip_step, region, col2plt, slipsybsz, "rainbow",
                                     reverse_cbar=False)

    fig.show()

    if flag_savefig:
        figname = meshname + "_inter_and_coseis_preferred.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxslip = 2.8
    slip_step = 0.2
    slip_step = maxslip/20
else:
    # maxslip = 2.8  # used for gamma=6e2
    maxslip = 3.6  # used for gamma=2.5e2/3e2
    # slip_step = 0.2
    # slip_step = maxslip/20
    slip_step = maxslip/10

potency_col = 'potency'

# plot_inferred_locking_and_slip(m_s_3d_ori_lock, mom_rate_3d_ori, m_s_hom, mom_hom,
#                               'locking', 'mag', region1, maxslip, slip_step, flag_savefig=flag_savefig)
plot_inferred_locking_and_slip(m_s_3d_ori_lock, mom_rate_3d_ori, m_s_3d_ori, mom_3d_ori,
                              'locking', 'mag', region2, maxslip, slip_step, flag_savefig=flag_savefig)
# m_s_3d_lock, mom_rate_3d, m_s_3d, mom_3d,
#                                    col2plt_lock, col2plt, region, maxslip, slip_step, flag_savefig=False):

In [ ]:
def plot_all_inferred_slip_and_diff_extra2(m_s_hom, m_s_1d, m_s_3d, 
                                            m_s_hom_lock, m_s_1d_lock, m_s_3d_lock,
                                            mom_hom, mom_1d, mom_3d, 
                                            col2plt, region, maxslip, slip_step, maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    colormap_diff = "roma"

    mu_model_strings = ["Ampl. 3D", "1D", "Ampl. 3D - H", "1D - H"]

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.25c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0

        # Slip inferred from the 3D model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_slip_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # Slip inferred from the 1D model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_slip_panel(fig, m_s_1d, m_s_1d_lock, mom_1d, panel_idx, mu_model_strings[panel_idx], 
                                     panel_labels[panel_idx], maxslip, slip_step, region, col2plt, slipsybsz, colormap)

        # difference between the 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 0]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_3d, m_s_3d_lock, mom_3d, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

        # difference between the 1D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_slip_diff_panel(fig, m_s_1d, m_s_1d_lock, mom_1d, mu_model_strings[panel_idx], 
                                          panel_labels[panel_idx], maxdiff, diff_step, region, col2plt, 
                                          slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_coseis_and_diff.pdf"
        figname = meshname + "_ref1D_suppl2_coseis_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxslip = 2.8
    slip_step = 0.2
    slip_step = maxslip/20

    maxdiff = 0.4
    # diff_step = 0.05
    diff_step = maxdiff/10

else:
    # maxslip = 2.8  # used for gamma=6e2
    maxslip = 3.6  # used for gamma=2.5e2/3e2
    # slip_step = 0.2
    # slip_step = maxslip/20
    slip_step = maxslip/10

    # maxdiff = 0.3  # used for gamma=6e2
    maxdiff = 0.5  # used for gamma=2.5e2/3e2
    # diff_step = 0.05
    diff_step = maxdiff/10

plot_all_inferred_slip_and_diff_extra2(m_s_hom, m_s_1d, m_s_3d, 
                                        m_s_hom_lock, m_s_1d_lock, m_s_3d_lock,
                                        mom_hom, mom_1d, mom_3d, 
                                        'mag', region1, maxslip, slip_step, maxdiff, diff_step, flag_savefig=flag_savefig)


In [ ]:
def plot_homvshet_slip(m_s_hom, m_s, col2plt, region, struc_str_inv, flag_savefig=False):

    slipsybsz = "c0.09c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = 1
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the heterogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet. inv."])
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet - Hom"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference", "y+l(m)"]) #

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_hethomslip.pdf")


plot_homvshet_slip(m_s_hom, m_s_sw, 'mag', region, sw_str, flag_savefig=False)    
plot_homvshet_slip(m_s_hom, m_s_3d, 'mag', region, het3d_str, flag_savefig=False)    
plot_homvshet_slip(m_s_hom, m_s_3d_ori, 'mag', region, het3d_ori_str, flag_savefig=False)    


In [ ]:
rmse_hom = utp.rmse_3d_dataframe(u_pred_hom, u_obs)* 1000  # Convert to mm   
rmse_sw = utp.rmse_3d_dataframe(u_pred_sw, u_obs)* 1000  # Convert to mm   
rmse_3d = utp.rmse_3d_dataframe(u_pred_3d, u_obs)* 1000  # Convert to mm   
rmse_3d_ori = utp.rmse_3d_dataframe(u_pred_3d_ori, u_obs)* 1000  # Convert to mm   
print(rmse_hom, rmse_sw, rmse_3d, rmse_3d_ori)

In [ ]:
# # plot the histogram of difference in slip
# fig, ax = plt.subplots(figsize=(4, 4.5), dpi=300)

# xmin, xmax = -0.25, 0.25
# bin_width = 0.01
# bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
# median = np.median(m_s_mag['diff'])
# sigma = np.std(m_s_mag['diff'])
# # rounded_sigma = round(sigma)
# counts1, _, _ = ax.hist(m_s_mag['diff'], bins=bins, color='dimgray', alpha=0.7, edgecolor='black')
# ymin, ymax = 0, np.max(counts1) + 100
# ax.errorbar(median, ymax*0.82, xerr=sigma, fmt='o', color='black', capsize=5, capthick=2, 
#             linewidth=2, label=fr'$\pm1\sigma = \pm{sigma:.3f}$')
# ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median = {median:.3f}')
# ax.legend()
# ax.set_xlabel('Difference (mm)')
# ax.set_ylabel('Frequency')
# ax.set_title('Difference in magnitude (Het-Hom)', fontweight='bold')
# ax.set_xlim(xmin, xmax)
# ax.set_ylim(ymin, ymax)


In [ ]:
# for obs, pred vectors
m2cm = 1e2  # conversion factor from meters to centimeters
ref_cm = 20  # reference length in cm

# for error vectors
m2mm = 1e3  # conversion factor from meters to millimeters
ref_err_mm = 5   # reference length for error in mm

ref_rat = 10 # reference ratio for data and error vectors
# ref_rat = ref_cm / ref_err_cm  # ratio of reference lengths for data and error vectors

# A different way of constructing the vectors for plotting arrows
# convert from m to mm, horizontal components
df_obs_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_obs['ux']*m2cm/ref_cm,
        "north_velocity": u_obs['uy']*m2cm/ref_cm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)
df_obs_h.head()

np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

# vertical components
df_obs_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_obs['uz']*m2cm/ref_cm,
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)


In [ ]:
### data fitting by the heterogeneous inversion 
# convert from m to mm
df_pred_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_pred['ux']*m2cm/ref_cm,
        "north_velocity": u_pred['uy']*m2cm/ref_cm,        
    }
)
df_pred_h.head()

df_pred_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_pred['uz']*m2cm/ref_cm,        
    }
)

# for error vectors
ref_res_mm = 25   # reference length for error in mm
ref_err_mm = 5   # reference length for error in mm
ref_rat_res = 5 # reference ratio for data and error vectors

# convert from m to mm, in order to directly compare with 
df_res_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_res['ux']*m2mm/ref_res_mm,
        "north_velocity": u_res['uy']*m2mm/ref_res_mm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,        
    }
)
df_res_h.head()
df_res_h_dum = df_res_h.iloc[:, :4]

df_res_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_res['uz']*m2mm/ref_res_mm,    
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,     
    }
)
df_res_v_dum = df_res_v.iloc[:, :4]


In [ ]:
# plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_h, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat_res, "north_sigma": 1/ref_rat_res, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")

fig.show()


In [ ]:
### data fitting by the homogeneous inversion 
# convert from m to mm
df_pred_hom_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_pred_hom['ux']*m2cm/ref_cm,
        "north_velocity": u_pred_hom['uy']*m2cm/ref_cm,       
    }
)
df_pred_h.head()

df_pred_hom_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_pred_hom['uz']*m2cm/ref_cm,      
    }
)

# convert from m to mm, in order to directly compare with 
df_res_hom_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_res_hom['ux']*m2mm/ref_res_mm,
        "north_velocity": u_res_hom['uy']*m2mm/ref_res_mm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,        
    }
)
df_res_hom_h.head()
df_res_hom_h_dum = df_res_hom_h.iloc[:, :4]

df_res_hom_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_res_hom['uz']*m2mm/ref_res_mm,   
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,     
    }
)
df_res_hom_v_dum = df_res_hom_v.iloc[:, :4]


In [ ]:
# plot the predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_h, pen="0.5p,blue", uncertaintyfill=None, line="0.25p,blue", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat_res, "north_sigma": 1/ref_rat_res, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")


fig.show()


In [ ]:
# Plot the comparison on horizontal components
fig = pygmt.Figure()
with fig.subplot(nrows=2, ncols=3, figsize=("18c", "12c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # homogeneous Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")        

    # heterogeneous Observed vs. predicted
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

    # residuals, hom vs het
    with fig.set_panel(panel=[0, 2]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_h_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        fig.velo(data=df_res_h_dum, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres_hom = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        lgdres = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

    ### Plot the comparison on vectical components
    # homogeneous Observed vs. predicted
    with fig.set_panel(panel=[1, 0]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_v, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")      

    # heterogeneous Observed vs. predicted
    with fig.set_panel(panel=[1, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_v, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")

    # residuals, hom vs het
    with fig.set_panel(panel=[1, 2]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_v_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        fig.velo(data=df_res_v_dum, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres_hom = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        lgdres = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

fig.show()  

In [ ]:
# plot the histogram of difference in residuals
fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.5), dpi=300)

bin_width = 2

ax = axes[0]

xmin, xmax = 0, 50 
bins = np.arange(xmin, xmax + bin_width, bin_width)
counts1, _, _ = ax.hist(u_res_hom['mag']*1000, bins=bins, color='blue', alpha=0.7, edgecolor='black',label='Homogeneous')
counts2, _, _ = ax.hist(u_res['mag']*1000, bins=bins, color='red', alpha=0.7, edgecolor='black',label='Heterogeneous')

ymin, ymax = 0, max(np.max(counts1), np.max(counts2)) + 4

median_hom = np.median(u_res_hom['mag']*1000)
sigma_hom = np.std(u_res_hom['mag']*1000)
median = np.median(u_res['mag']*1000)
sigma = np.std(u_res['mag']*1000)

rmse_hom = ut.rmse_3d_dataframe(u_pred_hom, u_obs)* 1000  # Convert to mm
print(f"HOM Residual RMSE: {rmse_hom:.2f} mm")
print(f"HOM Residual median: {median_hom:.2f} mm, sigma: {sigma_hom:.2f} mm")

rmse = ut.rmse_3d_dataframe(u_pred, u_obs)* 1000  # Convert to mm
print(f"HET Residual RMSE: {rmse:.2f} mm")
print(f"HET Residual median: {median:.2f} mm, sigma: {sigma:.2f} mm")

ax.plot([median_hom, median_hom], [ymin, ymax], '--', color='blue', label=fr'Hom median={median_hom:.1f}')
ax.plot([median, median], [ymin, ymax], '--', color='red', label=fr'Het median={median:.1f}')
ax.errorbar(median_hom, ymax-2, xerr=sigma_hom, fmt='o', color='blue', capsize=5, capthick=2, 
            linewidth=2, label=fr'Hom $\pm1\sigma=\pm{sigma_hom:.1f}$')
ax.errorbar(median, ymax-2, xerr=sigma, fmt='o', color='red', capsize=5, capthick=2, 
            linewidth=2, label=fr'Het $\pm1\sigma=\pm{sigma:.1f}$')
ax.text(0.1, 0.9, f"Hom Res RMSE:{rmse_hom:.2f}", fontsize=9, color='k', transform=ax.transAxes)
ax.text(0.1, 0.8, f"Het Res RMSE:{rmse:.2f}", fontsize=9, color='k', transform=ax.transAxes)

ax.legend(fontsize=9, loc='best')
ax.set_xlabel('Residual (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax+2, 2))
ax.set_xlim(xmin, xmax)


ax = axes[1]
bin_width = 1
xmin, xmax = -15, 15
bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
counts1, _, _ = ax.hist(u_res['mag']*1000-u_res_hom['mag']*1000, bins=bins, color='dimgray',alpha=0.7, edgecolor='black',label='het-hom')
ymin, ymax = 0, np.max(counts1) + 2

median = np.median(u_res['mag']*1000-u_res_hom['mag']*1000)
ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median={median:.2f}')

ax.legend(fontsize=9, loc='right')
ax.set_xlabel('Difference (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Residual difference (Het-Hom)', fontweight='bold')
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax+2, 2))
ax.set_xlim(xmin, xmax)